In [ ]:
# Gold log return
df["log_return"] = np.log(df["Gold"]).diff()

# Tomorrow's return as target
df["target"] = df["log_return"].shift(-1)

In [ ]:
exog_vars = ["X1","X2","X3","X4","X5","X6","X7","X8","X9"]

In [ ]:
lags = [1,2,3,5,10,20]

# Gold return lags
for lag in lags:
    df[f"gold_lag_{lag}"] = df["log_return"].shift(lag)

# Exogenous variable lags
for col in exog_vars:
    for lag in [1,2,5]:
        df[f"{col}_lag_{lag}"] = df[col].shift(lag)

In [ ]:
# Save last row BEFORE dropping NaNs
latest_row = df.iloc[-1:].copy()

# Remove last row from training
train_df = df.iloc[:-1]

In [ ]:
train_df = train_df.dropna()

In [ ]:
X_train = train_df.drop("target", axis=1)
y_train = train_df["target"]

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=800,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
latest_features = latest_row.drop("target", axis=1)

# Make sure column order matches training
latest_features = latest_features[X_train.columns]

In [ ]:
print("Missing values:", latest_features.isna().sum().sum())

In [ ]:
tomorrow_log_return = model.predict(latest_features)[0]

print("Predicted tomorrow log return:", tomorrow_log_return)

In [ ]:
last_price = df["Gold"].iloc[-1]

predicted_price = last_price * np.exp(tomorrow_log_return)

print("Predicted tomorrow price:", predicted_price)